# Where is fire risk high _right now_?

**Red Flag Warning + active fire detections + burn history**

This notebook answers an operational screening question: _of the places currently under a
Red Flag Warning, which ones already have fire on the ground and sit in a landscape that
burns repeatedly?_ It stacks three layers of very different velocity:

| Layer                      | Source                                         | Velocity | What it tells you                                                 |
| -------------------------- | ---------------------------------------------- | -------- | ----------------------------------------------------------------- |
| **Red Flag Warnings**      | NOAA/NWS active alerts API                     | minutes  | Where fire weather is happening _now_ (wind + low RH + dry fuels) |
| **Active fire detections** | NASA FIRMS (MODIS 24h, or VIIRS via free key)  | hours    | Where heat signatures are being detected _now_                    |
| **Burn history**           | MTBS perimeters 1984–2024 (Source Cooperative) | static   | Whether this landscape has a fire regime                          |

Endpoints are **read out of the project's dataset catalog**
(`analysis/fire_datasets.csv`) rather than hardcoded, so the notebook follows the catalog
if those rows change.

### Unit of analysis

The **NWS forecast zone** under warning. That is the unit the warning is actually issued
for, so no downscaling or areal interpolation is invented along the way.

### How the code is organised

Each step is a set of small single-purpose functions followed by a short orchestration
cell. Functions take their inputs as arguments rather than reading globals, return values
rather than mutating shared state, and handle empty input by returning empty frames — so
the "no active warnings" case falls out of the same code path instead of needing guards
everywhere. The last section runs a **smoke test** on synthetic fixtures, which works
offline and exercises the joins, the burn-history overlay, and the scoring.

### What the score is and is not

The composite is a **triage ranking**, not a calibrated probability. It sorts zones that
are _already_ under a Red Flag Warning by current fire activity and landscape fire
history. For calibrated burn probability, use the USFS Probabilistic Wildfire Risk or
Wildfire Risk to Communities rows in the same catalog — see the closing section.

### Requirements

`pandas geopandas shapely pyproj duckdb requests matplotlib` (optional: `folium`).
Everything here uses **no-auth** endpoints. A free FIRMS `MAP_KEY` is optional and
upgrades the detection layer from 1 km MODIS to 375 m VIIRS.


## Configuration

All tunables live in one frozen `Settings` object that gets passed into the functions that need it. Nothing below reads configuration off a global.


In [ ]:
from __future__ import annotations

import io
import json
import os
import time
import zipfile
from dataclasses import dataclass, field, replace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

import geopandas as gpd
from shapely import from_wkb
from shapely.geometry import shape

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 160)


@dataclass(frozen=True)
class Settings:
    """Everything tunable about a run."""

    # which alerts count as "fire weather is happening now"
    # add "Fire Weather Watch" to widen the net (a Watch is a heads-up, not a warning)
    alert_events: tuple[str, ...] = ("Red Flag Warning",)

    # detections: free key at https://firms.modaps.eosdis.nasa.gov/api/ -> 375 m VIIRS
    firms_map_key: str = ""
    firms_product: str = "VIIRS_NOAA20_NRT"
    detection_days: int = 1
    modis_min_confidence: int = 30  # MODIS confidence is 0-100
    viirs_keep_confidence: frozenset = frozenset({"n", "h"})  # VIIRS is l/n/h

    # burn history
    wildfire_only: bool = True  # MTBS also contains prescribed fire
    bbox_pad_deg: float = 0.5

    # geometry
    nearby_buffer_km: float = 25.0
    equal_area: str = "EPSG:5070"  # CONUS Albers; see caveats for AK/HI/PR

    # plumbing — NWS throttles generic user agents, put a real contact here
    nws_user_agent: str = (
        "ptps-wildfire-demo/0.2 (https://github.com/investinopen/ptps-wildfire-demo)"
    )
    cache_dir: Path = Path("data_cache")

    @property
    def nws_headers(self) -> dict[str, str]:
        return {"User-Agent": self.nws_user_agent, "Accept": "application/geo+json"}


@dataclass(frozen=True)
class Endpoints:
    """URLs resolved from the dataset catalog."""

    catalog: str
    alerts: str
    firms_zip: str
    mtbs: str
    firms_csv_fallback: str = (
        "https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/csv/"
        "MODIS_C6_1_USA_contiguous_and_Hawaii_24h.csv"
    )


CATALOG_URL = (
    "https://raw.githubusercontent.com/investinopen/"
    "ptps-wildfire-demo/main/analysis/fire_datasets.csv"
)

WEIGHTS = {
    "detections": 0.35,  # fire already burning inside the zone
    "intensity": 0.15,  # summed FRP - a crude intensity proxy
    "nearby_detections": 0.15,  # fire just outside the boundary
    "burned_share": 0.20,  # share of the zone burned since 1984
    "fire_frequency": 0.15,  # count of distinct historical fires
}

CFG = Settings(firms_map_key=os.environ.get("FIRMS_MAP_KEY", "").strip())
CFG.cache_dir.mkdir(exist_ok=True)

RUN_AT = datetime.now(timezone.utc)
print("Run time (UTC):", RUN_AT.isoformat(timespec="seconds"))
print("Alert events  :", ", ".join(CFG.alert_events))
print(
    "FIRMS key     :", "set" if CFG.firms_map_key else "not set (using no-auth MODIS)"
)

Run time (UTC): 2026-09-10T13:58:18+00:00
Alert events  : Red Flag Warning
FIRMS key     : not set (using no-auth MODIS)


## Utilities

Caching HTTP, plus two helpers that show up in every step: a normaliser for skewed counts and a tolerant column matcher.


In [ ]:
def cache_is_fresh(path: Path, ttl_minutes: float | None) -> bool:
    """True if `path` exists and is young enough. ttl_minutes=None means never expire."""
    if not path.exists():
        return False
    if ttl_minutes is None:
        return True
    return (time.time() - path.stat().st_mtime) / 60 < ttl_minutes


def read_cache(path: Path, binary: bool):
    return path.read_bytes() if binary else path.read_text(encoding="utf-8")


def write_cache(path: Path, response: requests.Response, binary: bool) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if binary:
        path.write_bytes(response.content)
    else:
        path.write_text(response.text, encoding="utf-8")


def http_get(
    url,
    *,
    cache: Path | None = None,
    ttl_minutes=None,
    headers=None,
    binary=False,
    timeout=120,
    retries=3,
):
    """GET with optional on-disk caching. `cache` is a full path, not a name."""
    if cache is not None and cache_is_fresh(cache, ttl_minutes):
        return read_cache(cache, binary)

    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers or {}, timeout=timeout)
            r.raise_for_status()
            if cache is not None:
                write_cache(cache, r, binary)
            return r.content if binary else r.text
        except Exception as exc:  # noqa: BLE001
            last = exc
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"GET failed after {retries} tries: {url}\n{last}")


def get_json(url, **kwargs):
    return json.loads(http_get(url, **kwargs))


def chunked(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i : i + n]

In [ ]:
def scale01(values, log: bool = False) -> pd.Series:
    """Min-max to [0,1]; log1p first for heavy-tailed counts. All-equal -> all zeros."""
    s = pd.to_numeric(pd.Series(values), errors="coerce").fillna(0.0).astype(float)
    if log:
        s = np.log1p(s.clip(lower=0))
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)


def pick_column(available, *candidates) -> str | None:
    """Case-insensitive column matcher, tolerant of upstream schema drift.

    Tries exact matches first, then substring matches, in candidate order.
    """
    lower = {c.lower(): c for c in available}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    for cand in candidates:
        for low, orig in lower.items():
            if cand.lower() in low:
                return orig
    return None


def empty_gdf(columns, crs="EPSG:4326") -> gpd.GeoDataFrame:
    """An empty GeoDataFrame with a known schema, so downstream code needs no guards."""
    df = pd.DataFrame(
        {c: pd.Series(dtype="object") for c in columns if c != "geometry"}
    )
    df["geometry"] = gpd.GeoSeries([], dtype="geometry")
    return gpd.GeoDataFrame(df, geometry="geometry", crs=crs)


ZONE_COLUMNS = ["zone_id", "zone_name", "state", "zone_type", "geometry"]

## Step 0 — Resolve endpoints from the dataset catalog

The catalog is the source of truth for _where the data lives_. Three rows carry this notebook.


In [4]:
def load_catalog(url: str) -> pd.DataFrame:
    return pd.read_csv(url)


def catalog_row(catalog: pd.DataFrame, name_fragment: str) -> pd.Series:
    hit = catalog[catalog["name"].str.contains(name_fragment, case=False, na=False)]
    if hit.empty:
        raise KeyError(f"No catalog row matching {name_fragment!r}")
    return hit.iloc[0]


def resolve_endpoints(catalog: pd.DataFrame, catalog_url: str) -> Endpoints:
    return Endpoints(
        catalog=catalog_url,
        alerts=catalog_row(catalog, "NOAA weather alerts")["example_data_url"],
        firms_zip=catalog_row(catalog, "FIRMS KML fire footprints")["example_data_url"],
        mtbs=catalog_row(catalog, "cboettig fire")["example_data_url"],
    )


def describe_sources(catalog: pd.DataFrame, fragments) -> pd.DataFrame:
    rows = [catalog_row(catalog, f) for f in fragments]
    return pd.DataFrame(rows)[
        ["name", "description", "data_velocity", "access_type"]
    ].reset_index(drop=True)

In [ ]:
catalog = load_catalog(CATALOG_URL)
ENDPOINTS = resolve_endpoints(catalog, CATALOG_URL)

print(f"{len(catalog)} datasets in catalog\n")
display(
    describe_sources(
        catalog, ["NOAA weather alerts", "FIRMS KML fire footprints", "cboettig fire"]
    )
)

for label in ("alerts", "firms_zip", "mtbs"):
    print(f"{label:>10}: {getattr(ENDPOINTS, label)}")

25 datasets in catalog



,name,description,data_velocity,access_type
0,NOAA weather alerts,"Forecasts and active alerts, including Red Flag Warnings.",high,No auth
1,NASA FIRMS KML fire footprints,Regional active-fire footprint polygons.,high,No auth
2,cboettig fire,Historical burned-area perimeters and hazard potential.,low,No auth


    alerts: https://api.weather.gov/alerts/active?event=Red%20Flag%20Warning
 firms_zip: https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/shapes/zips/MODIS_C6_1_USA_contiguous_and_Hawaii_24h.zip
      mtbs: https://data.source.coop/cboettig/fire/mtbs-perimeters-1984-2024.parquet


## Step 1 — Red Flag Warnings (the "right now" layer)

The NWS alerts endpoint returns CAP alerts wrapped in GeoJSON. Two things matter here:

1. Most Red Flag Warnings carry **`geometry: null`**. They are issued _for a zone_, not
   for an ad-hoc polygon, so the shape has to be resolved from `affectedZones`. Code that
   reads `feature["geometry"]` silently drops most of the warnings.
2. The same zone can be covered by more than one alert, so alerts are exploded to
   zone level and re-aggregated.

Fetching, parsing, and geometry resolution are separate functions — the parser is pure and
testable against a saved feature, with no network involved.


In [ ]:
def alerts_url(event: str) -> str:
    return f"https://api.weather.gov/alerts/active?event={requests.utils.quote(event)}"


def fetch_active_alerts(cfg: Settings, events=None) -> list[dict]:
    """Fetch raw GeoJSON alert features for each event type."""
    features = []
    for event in events or cfg.alert_events:
        js = get_json(
            alerts_url(event),
            cache=cfg.cache_dir / f"alerts_{event.replace(' ', '_')}.json",
            ttl_minutes=10,
            headers=cfg.nws_headers,
        )
        feats = js.get("features", [])
        print(f"{event}: {len(feats)} active alerts")
        features.extend(feats)
    return features


def zone_id_from_url(url: str) -> str:
    return url.rstrip("/").split("/")[-1]


def parse_alert(feature: dict) -> dict:
    """One GeoJSON alert feature -> one flat record. Pure; no network."""
    p = feature.get("properties", {})
    headlines = p.get("parameters", {}).get("NWSheadline") or [p.get("headline")]
    zone_urls = p.get("affectedZones") or []
    return {
        "alert_id": p.get("id"),
        "event": p.get("event"),
        "severity": p.get("severity"),
        "urgency": p.get("urgency"),
        "certainty": p.get("certainty"),
        "area_desc": p.get("areaDesc"),
        "sender": p.get("senderName"),
        "effective": pd.to_datetime(p.get("effective"), utc=True, errors="coerce"),
        "expires": pd.to_datetime(p.get("expires"), utc=True, errors="coerce"),
        "ends": pd.to_datetime(p.get("ends"), utc=True, errors="coerce"),
        "headline": headlines[0] if headlines else None,
        "zones": [zone_id_from_url(z) for z in zone_urls],
        "zone_urls": zone_urls,
        "ugc": (p.get("geocode") or {}).get("UGC", []),
        "has_own_geometry": feature.get("geometry") is not None,
    }


def alerts_to_frame(features: list[dict]) -> pd.DataFrame:
    columns = list(parse_alert({}).keys())
    if not features:
        return pd.DataFrame({c: pd.Series(dtype="object") for c in columns})
    return pd.DataFrame([parse_alert(f) for f in features])


def zone_urls_by_id(alerts: pd.DataFrame) -> dict[str, str]:
    """Unique zone id -> one canonical URL (which encodes the zone type)."""
    mapping: dict[str, str] = {}
    if alerts.empty:
        return mapping
    for urls in alerts["zone_urls"]:
        for u in urls or []:
            mapping.setdefault(zone_id_from_url(u), u)
    return mapping

In [ ]:
def fetch_zone_batch(cfg: Settings, zone_ids) -> dict[str, dict]:
    """Resolve up to ~40 zones in one call via /zones?id=...&include_geometry=true."""
    batch = list(zone_ids)
    if not batch:
        return {}
    url = "https://api.weather.gov/zones?include_geometry=true&id=" + ",".join(batch)
    key = abs(hash(tuple(batch))) % (10**12)
    js = get_json(
        url,
        cache=cfg.cache_dir / f"zones_{key}.json",
        ttl_minutes=60 * 24 * 30,
        headers=cfg.nws_headers,
    )
    return {
        f["properties"]["id"]: f
        for f in js.get("features", [])
        if f.get("properties", {}).get("id")
    }


def fetch_zone_single(cfg: Settings, zone_id: str, url: str) -> dict | None:
    """Fallback for zones the batch endpoint missed."""
    feature = get_json(
        url,
        cache=cfg.cache_dir / f"zone_{zone_id}.json",
        ttl_minutes=60 * 24 * 30,
        headers=cfg.nws_headers,
    )
    return feature if feature.get("geometry") is not None else None


def zone_features_to_gdf(features) -> gpd.GeoDataFrame:
    """GeoJSON zone features -> GeoDataFrame. Drops zones without geometry."""
    rows = []
    for f in features:
        if f.get("geometry") is None:
            continue
        p = f.get("properties", {})
        rows.append(
            {
                "zone_id": p.get("id"),
                "zone_name": p.get("name"),
                "state": p.get("state"),
                "zone_type": p.get("type"),
                "geometry": shape(f["geometry"]),
            }
        )
    if not rows:
        return empty_gdf(ZONE_COLUMNS)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")


def fetch_zone_geometries(
    cfg: Settings, url_by_id: dict[str, str], batch_size: int = 40
) -> gpd.GeoDataFrame:
    """Batch first, then fill gaps one at a time. Failures degrade, they don't raise."""
    ids = sorted(url_by_id)
    if not ids:
        return empty_gdf(ZONE_COLUMNS)

    found: dict[str, dict] = {}
    for batch in chunked(ids, batch_size):
        try:
            found.update(fetch_zone_batch(cfg, batch))
        except Exception as exc:  # noqa: BLE001
            print(f"  batch lookup failed ({len(batch)} zones): {exc}")

    missing = [z for z in ids if z not in found or found[z].get("geometry") is None]
    if missing:
        print(f"  resolving {len(missing)} zones individually…")
    for zone_id in missing:
        try:
            feature = fetch_zone_single(cfg, zone_id, url_by_id[zone_id])
            if feature is not None:
                found[zone_id] = feature
        except Exception as exc:  # noqa: BLE001
            print(f"  zone {zone_id}: {exc}")

    gdf = zone_features_to_gdf(found.values())
    dropped = len(ids) - len(gdf)
    if dropped:
        print(f"  {dropped} zone(s) had no geometry and were dropped")
    return gdf

In [ ]:
def aggregate_alerts_by_zone(alerts: pd.DataFrame) -> pd.DataFrame:
    """Explode alerts to zone level, then collapse back to one row per zone."""
    columns = [
        "zone_id",
        "n_alerts",
        "events",
        "severity",
        "expires",
        "headline",
        "offices",
    ]
    if alerts.empty:
        return pd.DataFrame({c: pd.Series(dtype="object") for c in columns})

    long = (
        alerts.explode("zones")
        .rename(columns={"zones": "zone_id"})
        .dropna(subset=["zone_id"])
    )
    return (
        long.groupby("zone_id")
        .agg(
            n_alerts=("alert_id", "nunique"),
            events=("event", lambda s: ", ".join(sorted(set(s)))),
            severity=("severity", "first"),
            expires=("expires", "max"),
            headline=("headline", "first"),
            offices=("sender", lambda s: ", ".join(sorted(set(s.dropna())))),
        )
        .reset_index()
    )


def build_warned_zones(
    alerts: pd.DataFrame, zone_geoms: gpd.GeoDataFrame, run_at: datetime
) -> gpd.GeoDataFrame:
    """Join zone polygons to their alert attributes; add time left on the warning."""
    if alerts.empty or zone_geoms.empty:
        return empty_gdf(
            ZONE_COLUMNS
            + [
                "n_alerts",
                "events",
                "severity",
                "expires",
                "headline",
                "offices",
                "hours_remaining",
            ]
        )
    warned = zone_geoms.merge(
        aggregate_alerts_by_zone(alerts), on="zone_id", how="inner"
    )
    warned["hours_remaining"] = (
        (warned["expires"] - run_at).dt.total_seconds() / 3600
    ).round(1)
    return warned


def summarize_warned(warned: gpd.GeoDataFrame) -> pd.DataFrame:
    if warned.empty:
        return pd.DataFrame(columns=["zones", "median_hours_left"])
    return (
        warned.groupby("state")
        .agg(
            zones=("zone_id", "count"), median_hours_left=("hours_remaining", "median")
        )
        .sort_values("zones", ascending=False)
    )

In [ ]:
alert_features = fetch_active_alerts(CFG)
alerts = alerts_to_frame(alert_features)
zone_geoms = fetch_zone_geometries(CFG, zone_urls_by_id(alerts))
warned = build_warned_zones(alerts, zone_geoms, RUN_AT)

if alerts.empty:
    print("\nNo active alerts for the selected event types.")
else:
    print(
        f"\n{len(alerts)} alerts -> {alerts['zones'].explode().nunique()} unique zones"
    )
    print(
        f"alerts carrying their own polygon: {int(alerts['has_own_geometry'].sum())} "
        f"({alerts['has_own_geometry'].mean():.0%}) "
        "— the rest are zone-referenced and need geometry resolved"
    )
    print(
        f"{len(warned)} zones under warning across {warned['state'].nunique()} states\n"
    )
    display(summarize_warned(warned).head(15))

Red Flag Warning: 11 active alerts


## Step 2 — Active fire detections

Two paths, same downstream schema:

- **No auth** — the FIRMS 24-hour MODIS file for CONUS + Hawaii listed in the catalog
  (1 km pixels).
- **Free key** — the FIRMS Area API with VIIRS (375 m), which sees smaller fires. Set
  `FIRMS_MAP_KEY` in the environment to take this path.

Loaders are separate from the normaliser, so a new product only needs a loader that
returns lat/lon columns. A detection is a **thermal anomaly**, not a confirmed wildfire:
flares, industrial heat, and agricultural burns all show up.


In [ ]:
def bbox_with_pad(gdf: gpd.GeoDataFrame, pad_deg: float) -> tuple[float, ...]:
    minx, miny, maxx, maxy = gdf.to_crs("EPSG:4326").total_bounds
    return (minx - pad_deg, miny - pad_deg, maxx + pad_deg, maxy + pad_deg)


def firms_area_url(cfg: Settings, bbox) -> str:
    return (
        f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{cfg.firms_map_key}/"
        f"{cfg.firms_product}/"
        + ",".join(f"{v:.3f}" for v in bbox)
        + f"/{cfg.detection_days}"
    )


def points_from_latlon(df: pd.DataFrame) -> gpd.GeoDataFrame:
    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    )


def load_firms_csv(url: str, cache: Path, ttl_minutes: float) -> gpd.GeoDataFrame:
    text = http_get(url, cache=cache, ttl_minutes=ttl_minutes)
    return points_from_latlon(pd.read_csv(io.StringIO(text)))


def load_firms_shapefile_zip(url: str, cache: Path) -> gpd.GeoDataFrame:
    raw = http_get(url, cache=cache, ttl_minutes=60, binary=True)
    name = next(
        (
            n
            for n in zipfile.ZipFile(io.BytesIO(raw)).namelist()
            if n.lower().endswith(".shp")
        ),
        None,
    )
    if name is None:
        raise RuntimeError("no .shp inside FIRMS archive")
    return gpd.read_file(f"zip://{cache.as_posix()}!{name}")


def filter_by_confidence(fires: gpd.GeoDataFrame, cfg: Settings) -> gpd.GeoDataFrame:
    """MODIS confidence is 0-100; VIIRS is l/n/h. Handle both, or neither."""
    if "confidence" not in fires.columns:
        return fires
    conf = fires["confidence"]
    if pd.api.types.is_numeric_dtype(conf):
        keep = (
            pd.to_numeric(conf, errors="coerce").fillna(0) >= cfg.modis_min_confidence
        )
    else:
        keep = conf.astype(str).str.lower().str[0].isin(cfg.viirs_keep_confidence)
    print(f"{int((~keep).sum()):,} detections dropped on confidence")
    return fires[keep].copy()


def normalize_detections(fires: gpd.GeoDataFrame, cfg: Settings) -> gpd.GeoDataFrame:
    """Lowercase columns, guarantee an `frp` column, filter, and set CRS."""
    fires = fires.copy()
    fires.columns = [c.lower() for c in fires.columns]
    if "frp" not in fires.columns:
        fires["frp"] = np.nan  # fire radiative power, MW - rough intensity proxy
    fires = filter_by_confidence(fires, cfg)
    return fires.set_geometry("geometry").to_crs("EPSG:4326")

In [ ]:
def load_detections(
    cfg: Settings, endpoints: Endpoints, warned: gpd.GeoDataFrame
) -> tuple[gpd.GeoDataFrame, str]:
    """Try keyed VIIRS, then the no-auth shapefile, then the plain CSV mirror."""
    if cfg.firms_map_key and not warned.empty:
        try:
            bbox = bbox_with_pad(warned, 1.0)
            fires = load_firms_csv(
                firms_area_url(cfg, bbox),
                cfg.cache_dir / "firms_viirs_area.csv",
                60,
            )
            note = f"FIRMS Area API — {cfg.firms_product}, {cfg.detection_days}d, 375 m"
            return normalize_detections(fires, cfg), note
        except Exception as exc:  # noqa: BLE001
            print(
                f"Keyed FIRMS request failed ({exc}); falling back to the no-auth file."
            )

    try:
        fires = load_firms_shapefile_zip(
            endpoints.firms_zip, cfg.cache_dir / "firms_modis_24h.zip"
        )
        note = "FIRMS MODIS C6.1 24h, CONUS+HI shapefile, 1 km"
    except Exception as exc:  # noqa: BLE001
        print(f"Shapefile read failed ({exc}); trying the plain CSV mirror.")
        fires = load_firms_csv(
            endpoints.firms_csv_fallback, cfg.cache_dir / "firms_modis_24h.csv", 60
        )
        note = "FIRMS MODIS C6.1 24h, CONUS+HI CSV, 1 km"
    return normalize_detections(fires, cfg), note

In [ ]:
fires, source_note = load_detections(CFG, ENDPOINTS, warned)

print(f"source: {source_note}")
print(f"{len(fires):,} detections retained")
preview = [
    c
    for c in [
        "latitude",
        "longitude",
        "acq_date",
        "acq_time",
        "confidence",
        "frp",
        "daynight",
    ]
    if c in fires.columns
]
display(fires.head(5)[preview])

## Step 3 — Burn history (MTBS perimeters, 1984–2024)

The MTBS perimeter file is large, so it is queried remotely with DuckDB rather than
downloaded whole: column pruning plus a bounding-box predicate means only the row groups
intersecting the warned area come over the wire.

Two schema details get their own functions because they are the usual failure points:

- `resolve_mtbs_columns` matches names case-insensitively (`Ig_Date`, `BurnBndAc`, …).
- `geoparquet_crs` reads the CRS from the file's `geo` metadata key. MTBS ships in an
  Albers projection natively but cloud-optimised republications are often reprojected to
  lon/lat, and getting this wrong silently produces empty intersections rather than an
  error.


In [ ]:
import duckdb


@dataclass(frozen=True)
class MtbsColumns:
    """Where the fields we need actually live in this particular parquet."""

    id: str | None
    name: str | None
    type: str | None
    acres: str | None
    date: str | None
    lat: str | None
    lon: str | None
    geometry: str | None
    bbox: str | None

    def selected(self) -> list[str]:
        return [
            c
            for c in (
                self.id,
                self.name,
                self.type,
                self.acres,
                self.date,
                self.lat,
                self.lon,
                self.geometry,
            )
            if c
        ]


def open_duckdb() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    return con


def parquet_schema(con, url: str) -> pd.DataFrame:
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{url}')").fetch_df()


def resolve_mtbs_columns(columns) -> MtbsColumns:
    return MtbsColumns(
        id=pick_column(columns, "Event_ID", "event_id", "id"),
        name=pick_column(columns, "Incid_Name", "incid_name", "name"),
        type=pick_column(columns, "Incid_Type", "incid_type", "type"),
        acres=pick_column(columns, "BurnBndAc", "acres"),
        date=pick_column(columns, "Ig_Date", "ig_date", "date"),
        lat=pick_column(columns, "BurnBndLat", "lat"),
        lon=pick_column(columns, "BurnBndLon", "lon"),
        geometry=pick_column(columns, "geometry", "geom", "wkb"),
        bbox=pick_column(columns, "bbox"),
    )


def geoparquet_crs(con, url: str, geometry_column: str | None):
    """Read the CRS out of the GeoParquet `geo` metadata key. None if unavailable."""
    try:
        rows = con.execute(
            f"SELECT key, value FROM parquet_kv_metadata('{url}')"
        ).fetchall()
    except Exception as exc:  # noqa: BLE001
        print(f"could not read geo metadata: {exc}")
        return None

    for key, value in rows:
        key_s = key.decode() if isinstance(key, (bytes, bytearray)) else str(key)
        if key_s != "geo":
            continue
        val = value.decode() if isinstance(value, (bytes, bytearray)) else str(value)
        meta = json.loads(val)
        primary = meta.get("primary_column", geometry_column)
        spec = meta.get("columns", {}).get(primary, {}).get("crs")
        if isinstance(spec, dict):
            from pyproj import CRS as PyCRS

            return PyCRS.from_json_dict(spec)
        if isinstance(spec, str):
            from pyproj import CRS as PyCRS

            return PyCRS.from_user_input(spec)
        return "OGC:CRS84"  # GeoParquet default when crs is absent or null
    return None

In [ ]:
def bbox_predicate(cols: MtbsColumns, bounds, pad: float) -> str:
    """Cheapest available spatial prefilter: bbox struct > centroid columns > nothing."""
    minx, miny, maxx, maxy = bounds
    if cols.bbox:
        return (
            f'"{cols.bbox}".xmin <= {maxx + pad} AND "{cols.bbox}".xmax >= {minx - pad} '
            f'AND "{cols.bbox}".ymin <= {maxy + pad} AND "{cols.bbox}".ymax >= {miny - pad}'
        )
    if cols.lat and cols.lon:
        return (
            f'"{cols.lon}" BETWEEN {minx - pad} AND {maxx + pad} '
            f'AND "{cols.lat}" BETWEEN {miny - pad} AND {maxy + pad}'
        )
    print("No bbox or lat/lon columns — reading the full file. This will be slow.")
    return "TRUE"


def query_mtbs(con, url: str, cols: MtbsColumns, bounds, pad: float) -> pd.DataFrame:
    select = ", ".join(f'"{c}"' for c in cols.selected())
    sql = (
        f"SELECT {select} FROM read_parquet('{url}') "
        f"WHERE {bbox_predicate(cols, bounds, pad)}"
    )
    t0 = time.time()
    df = con.execute(sql).fetch_df()
    print(f"{len(df):,} perimeters returned in {time.time() - t0:.1f}s")
    return df


def decode_geometry(values) -> gpd.GeoSeries:
    """WKB blobs or WKT strings -> geometries."""
    values = list(values)
    first = next((v for v in values if v is not None), None)
    if first is None:
        return gpd.GeoSeries([], dtype="geometry")
    if isinstance(first, (bytes, bytearray, memoryview)):
        return gpd.GeoSeries(from_wkb(values))
    if isinstance(first, str):
        return gpd.GeoSeries.from_wkt(values)
    return gpd.GeoSeries(values)


def infer_crs(geoms: gpd.GeoSeries, equal_area: str) -> str:
    """Fallback when the file declares nothing: |x| > 180 means it can't be lon/lat."""
    bounds = geoms.head(200).bounds
    inferred = equal_area if bounds["minx"].abs().max() > 180 else "EPSG:4326"
    print(f"inferred CRS from coordinate range: {inferred}")
    return inferred


def build_mtbs_gdf(
    raw: pd.DataFrame, cols: MtbsColumns, crs, equal_area: str
) -> gpd.GeoDataFrame:
    geoms = decode_geometry(raw[cols.geometry])
    gdf = gpd.GeoDataFrame(
        {
            "event_id": raw[cols.id] if cols.id else np.arange(len(raw)),
            "incident": raw[cols.name] if cols.name else None,
            "incid_type": raw[cols.type] if cols.type else None,
            "acres": pd.to_numeric(raw[cols.acres], errors="coerce")
            if cols.acres
            else np.nan,
            "ig_date": pd.to_datetime(raw[cols.date], errors="coerce")
            if cols.date
            else pd.NaT,
        },
        geometry=geoms.values,
    )
    gdf.set_crs(
        crs if crs is not None else infer_crs(gdf.geometry, equal_area),
        inplace=True,
        allow_override=True,
    )
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf["ig_year"] = gdf["ig_date"].dt.year
    return gdf


def drop_prescribed(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if "incid_type" not in gdf.columns or not gdf["incid_type"].notna().any():
        return gdf
    keep = ~gdf["incid_type"].astype(str).str.contains(
        "prescribed", case=False, na=False
    )
    print(f"excluded {int((~keep).sum()):,} prescribed-fire records")
    return gdf[keep].copy()


def load_burn_history(
    cfg: Settings, endpoints: Endpoints, warned: gpd.GeoDataFrame
) -> gpd.GeoDataFrame:
    if warned.empty:
        print("No warned zones, so no burn history to pull.")
        return empty_gdf(
            [
                "event_id",
                "incident",
                "incid_type",
                "acres",
                "ig_date",
                "ig_year",
                "geometry",
            ]
        )

    con = open_duckdb()
    schema = parquet_schema(con, endpoints.mtbs)
    cols = resolve_mtbs_columns(schema["column_name"].tolist())
    print("resolved columns:", {k: v for k, v in cols.__dict__.items() if v})

    crs = geoparquet_crs(con, endpoints.mtbs, cols.geometry)
    print("declared CRS:", crs if crs is not None else "unknown (will infer)")

    raw = query_mtbs(
        con,
        endpoints.mtbs,
        cols,
        warned.to_crs("EPSG:4326").total_bounds,
        cfg.bbox_pad_deg,
    )
    gdf = build_mtbs_gdf(raw, cols, crs, cfg.equal_area)
    return drop_prescribed(gdf) if cfg.wildfire_only else gdf

In [ ]:
mtbs = load_burn_history(CFG, ENDPOINTS, warned)

if not mtbs.empty:
    print(
        f"\n{len(mtbs):,} historical perimeters near the warned area, "
        f"{mtbs['ig_year'].min():.0f}–{mtbs['ig_year'].max():.0f}"
    )
    display(mtbs.drop(columns="geometry").head())

## Step 4 — Combine the layers

Everything is projected to an equal-area CRS first, because the next steps measure areas
and distances. Three quantities are attached to each warned zone:

- **detections inside** the zone, and their summed fire radiative power
- **detections within `nearby_buffer_km`** of the zone — fire next door still matters
- **burn history**: share of the zone burned since 1984, number of distinct fires, and
  the year of the most recent one

The join and the summarise are separate functions so either can be swapped (for example,
counting detections in the last 6 hours only) without touching the other.


In [ ]:
def to_equal_area(gdf: gpd.GeoDataFrame, cfg: Settings) -> gpd.GeoDataFrame:
    return gdf.to_crs(cfg.equal_area)


def join_points_to_zones(
    points: gpd.GeoDataFrame, zones: gpd.GeoDataFrame, key: str = "zone_id"
) -> pd.DataFrame:
    if points.empty or zones.empty:
        return pd.DataFrame(columns=[key, "frp"])
    return gpd.sjoin(points, zones[[key, "geometry"]], how="inner", predicate="within")


def summarize_detections(joined: pd.DataFrame, key: str = "zone_id") -> pd.DataFrame:
    if joined.empty:
        return pd.DataFrame(columns=[key, "n_detections", "frp_sum"])
    return (
        joined.groupby(key)
        .agg(n_detections=("frp", "size"), frp_sum=("frp", "sum"))
        .reset_index()
    )


def buffer_zones(
    zones: gpd.GeoDataFrame, km: float, key: str = "zone_id"
) -> gpd.GeoDataFrame:
    ring = zones[[key, "geometry"]].copy()
    ring["geometry"] = ring.geometry.buffer(km * 1000)
    return ring


def attach_detection_stats(
    zones_ea: gpd.GeoDataFrame, fires_ea: gpd.GeoDataFrame, cfg: Settings
) -> gpd.GeoDataFrame:
    """Add n_detections, frp_sum, n_detections_nearby. Zero-filled, never NaN."""
    out = zones_ea.copy()
    out["zone_area_m2"] = out.geometry.area

    inside = summarize_detections(join_points_to_zones(fires_ea, out))
    ring = buffer_zones(out, cfg.nearby_buffer_km)
    nearby = (
        join_points_to_zones(fires_ea, ring)
        .groupby("zone_id")
        .size()
        .rename("n_detections_nearby")
        .reset_index()
        if not fires_ea.empty and not out.empty
        else pd.DataFrame(columns=["zone_id", "n_detections_nearby"])
    )

    out = out.merge(inside, on="zone_id", how="left").merge(
        nearby, on="zone_id", how="left"
    )
    cols = ["n_detections", "frp_sum", "n_detections_nearby"]
    out[cols] = out[cols].fillna(0)
    return out

In [ ]:
def repair_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """buffer(0) fixes the self-intersections that make overlay() throw."""
    out = gdf.copy()
    out["geometry"] = out.geometry.buffer(0)
    return out[out.geometry.notna() & ~out.geometry.is_empty]


def intersect_zones_with_fires(
    zones_ea: gpd.GeoDataFrame, mtbs_ea: gpd.GeoDataFrame
) -> gpd.GeoDataFrame:
    if zones_ea.empty or mtbs_ea.empty:
        return empty_gdf(["zone_id", "event_id", "ig_year"], crs=zones_ea.crs)
    t0 = time.time()
    inter = gpd.overlay(
        zones_ea[["zone_id", "geometry"]],
        mtbs_ea[["event_id", "ig_year", "geometry"]],
        how="intersection",
        keep_geom_type=True,
    )
    print(f"{len(inter):,} zone x fire intersections in {time.time() - t0:.1f}s")
    return inter


def summarize_burn_history(inter: gpd.GeoDataFrame) -> pd.DataFrame:
    """Counts plus dissolved burned area, so overlapping reburns aren't double counted."""
    columns = ["zone_id", "n_hist_fires", "last_burn_year", "burned_m2"]
    if inter.empty:
        return pd.DataFrame({c: pd.Series(dtype="float64") for c in columns})

    counts = (
        inter.groupby("zone_id")
        .agg(n_hist_fires=("event_id", "nunique"), last_burn_year=("ig_year", "max"))
        .reset_index()
    )
    burned = (
        inter.dissolve(by="zone_id").geometry.area.rename("burned_m2").reset_index()
    )
    return counts.merge(burned, on="zone_id", how="outer")


def attach_burn_history(
    zones_ea: gpd.GeoDataFrame, mtbs_ea: gpd.GeoDataFrame, run_year: int
) -> gpd.GeoDataFrame:
    stats = summarize_burn_history(intersect_zones_with_fires(zones_ea, mtbs_ea))
    out = zones_ea.merge(stats, on="zone_id", how="left")
    out["burned_m2"] = out["burned_m2"].fillna(0)
    out["n_hist_fires"] = out["n_hist_fires"].fillna(0)
    out["burned_frac"] = (out["burned_m2"] / out["zone_area_m2"]).clip(0, 1)
    out["years_since_burn"] = run_year - out["last_burn_year"]
    return out

In [ ]:
if warned.empty:
    zones_ea = warned
    print("No warned zones to analyse.")
else:
    zones_ea = attach_detection_stats(
        to_equal_area(warned, CFG), to_equal_area(fires, CFG), CFG
    )
    zones_ea = attach_burn_history(
        zones_ea,
        repair_geometries(to_equal_area(mtbs, CFG)) if not mtbs.empty else mtbs,
        RUN_AT.year,
    )

    n_hot = int((zones_ea["n_detections"] > 0).sum())
    print(f"\n{n_hot} of {len(zones_ea)} warned zones contain an active detection")
    print(
        f"{int(zones_ea['n_detections'].sum()):,} detections fall inside warned zones "
        f"({zones_ea['n_detections'].sum() / max(len(fires), 1):.1%} of all detections)"
    )
    print(
        f"median burned share of a warned zone since 1984: "
        f"{zones_ea['burned_frac'].median():.1%}"
    )

## Scoring

`score_components` turns raw quantities into comparable 0–1 components; `rank_zones`
applies the weights. Keeping them apart means you can inspect the components, reweight
without recomputing, or drop a component entirely by editing `WEIGHTS`.


In [ ]:
def score_components(zones: gpd.GeoDataFrame) -> pd.DataFrame:
    """Raw quantities -> comparable 0-1 components. Log scale for heavy-tailed counts."""

    def col(name):
        return (
            zones[name] if name in zones.columns else pd.Series(0.0, index=zones.index)
        )

    return pd.DataFrame(
        {
            "detections": scale01(col("n_detections"), log=True),
            "intensity": scale01(col("frp_sum"), log=True),
            "nearby_detections": scale01(col("n_detections_nearby"), log=True),
            "burned_share": scale01(col("burned_frac")),
            "fire_frequency": scale01(col("n_hist_fires"), log=True),
        },
        index=zones.index,
    )


def rank_zones(zones: gpd.GeoDataFrame, weights: dict) -> gpd.GeoDataFrame:
    """Weighted composite on a 0-100 scale, with each contribution kept as `c_*`."""
    if zones.empty:
        return zones
    out = zones.copy()
    components = score_components(out)
    for name, weight in weights.items():
        out[f"c_{name}"] = components[name] * weight
    out["score"] = (components * pd.Series(weights)).sum(axis=1) * 100
    out = out.sort_values("score", ascending=False).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out

In [ ]:
ranked = rank_zones(zones_ea, WEIGHTS)

if not ranked.empty:
    print(
        f"Scored {len(ranked)} warned zones. "
        f"Score range {ranked['score'].min():.1f}–{ranked['score'].max():.1f}."
    )
    print("Weights:", ", ".join(f"{k} {v:.0%}" for k, v in WEIGHTS.items()))

## Step 5 — The answer

Zones under a Red Flag Warning, ranked. Read the score as a queue order, not a
probability.

One deliberate asymmetry worth naming: `years_since_burn` is reported but **not scored**.
A landscape that burned two years ago is often _less_ dangerous in the short term because
the fuel is gone. Frequency and extent of past fire indicate a fire-prone regime; recency
cuts the other way. Folding both into one number would hide that tension, so recency stays
a column you read alongside the rank.


In [ ]:
TOP_COLUMNS = [
    "rank",
    "zone_id",
    "zone_name",
    "state",
    "score",
    "n_detections",
    "frp_sum",
    "n_detections_nearby",
    "burned_frac",
    "n_hist_fires",
    "years_since_burn",
    "hours_remaining",
]


def format_top_table(ranked: gpd.GeoDataFrame, n: int = 20) -> pd.DataFrame:
    cols = [c for c in TOP_COLUMNS if c in ranked.columns]
    top = ranked[cols].head(n).copy()
    top["score"] = top["score"].round(1)
    if "frp_sum" in top:
        top["frp_sum"] = top["frp_sum"].round(0)
    if "burned_frac" in top:
        top["burned_frac"] = (top["burned_frac"] * 100).round(1)
        top = top.rename(columns={"burned_frac": "burned_pct"})
    return top


def describe_hot_zones(ranked: gpd.GeoDataFrame, n: int = 5) -> list[str]:
    """One plain-language line per zone that is both warned and actively burning."""
    hot = ranked[ranked["n_detections"] > 0].head(n)
    return [
        f"{r.zone_name}, {r.state} ({r.zone_id}): {int(r.n_detections)} detections, "
        f"{getattr(r, 'burned_frac', 0):.0%} of the zone burned since 1984, "
        f"warning expires in {r.hours_remaining:.0f}h"
        for r in hot.itertuples()
    ]


def no_warnings_message(cfg: Settings, run_at: datetime) -> str:
    return (
        f"No zones under {' / '.join(cfg.alert_events)} at "
        f"{run_at.strftime('%Y-%m-%d %H:%M')} UTC.\n"
        "That is a real answer, not a failure — Red Flag Warnings are seasonal and often "
        "absent overnight. Try again during afternoon peak wind, or add "
        "'Fire Weather Watch' to Settings.alert_events."
    )


if ranked.empty:
    print(no_warnings_message(CFG, RUN_AT))
else:
    display(format_top_table(ranked))
    n_hot = int((ranked["n_detections"] > 0).sum())
    print(f"\n{n_hot} zones are under a warning AND have detections inside them.")
    for line in describe_hot_zones(ranked):
        print(f"  • {line}")

In [ ]:
import matplotlib.pyplot as plt

COMPONENT_COLORS = {
    "detections": "#b2182b",
    "intensity": "#ef8a62",
    "nearby_detections": "#fddbc7",
    "burned_share": "#4393c3",
    "fire_frequency": "#2166ac",
}


def zone_label(row) -> str:
    name = row.zone_name
    return f"{name[:28]} ({row.state})" if isinstance(name, str) else row.zone_id


def plot_score_breakdown(ranked, weights, run_at, events, n=15, ax=None):
    """Stacked bars showing which component put each zone where it is."""
    if ranked.empty:
        return None
    top = ranked.head(min(n, len(ranked))).iloc[::-1]
    labels = [zone_label(r) for r in top.itertuples()]

    if ax is None:
        _, ax = plt.subplots(figsize=(10, 0.42 * len(top) + 2))
    left = np.zeros(len(top))
    for name in weights:
        values = (top[f"c_{name}"] * 100).values
        ax.barh(
            labels,
            values,
            left=left,
            label=name.replace("_", " "),
            color=COMPONENT_COLORS[name],
            edgecolor="white",
            linewidth=0.5,
        )
        left += values

    ax.set_xlabel("composite score (0-100)")
    ax.set_title(
        f"Highest-priority zones under {' / '.join(events)}\n"
        f"{run_at.strftime('%Y-%m-%d %H:%M')} UTC",
        loc="left",
        fontsize=12,
    )
    ax.legend(loc="lower right", fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    return ax


if not ranked.empty:
    plot_score_breakdown(ranked, WEIGHTS, RUN_AT, CFG.alert_events)
    plt.tight_layout()
    plt.show()

In [ ]:
def clip_to_view(
    points: gpd.GeoDataFrame, frame: gpd.GeoDataFrame, pad_deg: float = 3.0
) -> gpd.GeoDataFrame:
    minx, miny, maxx, maxy = frame.total_bounds
    return points.cx[minx - pad_deg : maxx + pad_deg, miny - pad_deg : maxy + pad_deg]


def plot_zone_map(ranked, fires, events, source_note, ax=None):
    if ranked.empty:
        return None
    zones = ranked.to_crs("EPSG:4326")
    if ax is None:
        _, ax = plt.subplots(figsize=(13, 8))

    zones.plot(
        column="score",
        cmap="YlOrRd",
        linewidth=0.3,
        edgecolor="#777",
        legend=True,
        ax=ax,
        legend_kwds={"label": "composite score", "shrink": 0.6},
    )

    view = clip_to_view(fires.to_crs("EPSG:4326"), zones)
    view.plot(
        ax=ax,
        markersize=4,
        color="#111",
        alpha=0.6,
        label=f"active detections (n={len(view):,})",
    )

    ax.set_title(
        f"Zones under {' / '.join(events)}, shaded by composite score\n"
        f"black dots: FIRMS detections — {source_note}",
        loc="left",
        fontsize=12,
    )
    ax.legend(loc="lower left", frameon=False, fontsize=9)
    ax.set_axis_off()
    return ax


if not ranked.empty:
    plot_zone_map(ranked, fires, CFG.alert_events, source_note)
    plt.tight_layout()
    plt.show()

In [ ]:
MAP_FIELDS = ["zone_id", "zone_name", "state", "score", "n_detections", "n_hist_fires"]


def zone_style(feature, vmax):
    return {
        "fillColor": "#b2182b",
        "color": "#7f2704",
        "weight": 0.7,
        "fillOpacity": 0.15 + 0.6 * (feature["properties"]["score"] / vmax),
    }


def build_folium_map(ranked, fires, max_points: int = 3000):
    """Interactive version. Returns None if folium isn't installed or there's nothing to draw."""
    try:
        import folium
    except ImportError:
        print("folium not installed — skipping interactive map")
        return None
    if ranked.empty:
        return None

    zones = ranked.to_crs("EPSG:4326")
    centre = zones.geometry.union_all().centroid
    m = folium.Map(location=[centre.y, centre.x], zoom_start=5, tiles="OpenStreetMap")

    vmax = max(zones["score"].max(), 1e-9)
    folium.GeoJson(
        zones[MAP_FIELDS + ["geometry"]].to_json(),
        style_function=lambda f: zone_style(f, vmax),
        tooltip=folium.GeoJsonTooltip(
            fields=MAP_FIELDS[1:],
            aliases=["Zone", "State", "Score", "Detections", "Fires since 1984"],
        ),
    ).add_to(m)

    for _, row in (
        clip_to_view(fires.to_crs("EPSG:4326"), zones, 0).head(max_points).iterrows()
    ):
        folium.CircleMarker(
            [row.geometry.y, row.geometry.x],
            radius=2,
            color="#111",
            fill=True,
            fill_opacity=0.7,
            popup=f"FRP {row.get('frp', 'n/a')}",
        ).add_to(m)
    return m


fmap = build_folium_map(ranked, fires)
if fmap is not None:
    display(fmap)

In [ ]:
EXPORT_COLUMNS = [
    "rank",
    "zone_id",
    "zone_name",
    "state",
    "zone_type",
    "score",
    "n_detections",
    "frp_sum",
    "n_detections_nearby",
    "burned_frac",
    "n_hist_fires",
    "last_burn_year",
    "years_since_burn",
    "n_alerts",
    "events",
    "expires",
    "hours_remaining",
    "headline",
    "offices",
]


def build_provenance(
    cfg, endpoints, weights, source_note, ranked, fires, run_at
) -> dict:
    return {
        "run_at_utc": run_at.isoformat(),
        "catalog": endpoints.catalog,
        "alert_events": list(cfg.alert_events),
        "alerts_endpoint": endpoints.alerts,
        "detections_source": source_note,
        "burn_history": endpoints.mtbs,
        "weights": weights,
        "equal_area_crs": cfg.equal_area,
        "nearby_buffer_km": cfg.nearby_buffer_km,
        "n_zones": int(len(ranked)),
        "n_detections_total": int(len(fires)),
    }


def export_results(
    ranked, provenance: dict, run_at: datetime, out_dir: Path = Path(".")
) -> list[Path]:
    """Write CSV + GeoJSON + provenance. Returns the paths written."""
    if ranked.empty:
        return []
    out_dir.mkdir(parents=True, exist_ok=True)
    stamp = run_at.strftime("%Y%m%dT%H%M")
    cols = [c for c in EXPORT_COLUMNS if c in ranked.columns]

    csv_path = out_dir / f"fire_risk_ranked_{stamp}.csv"
    geo_path = out_dir / f"fire_risk_zones_{stamp}.geojson"
    prov_path = out_dir / f"fire_risk_provenance_{stamp}.json"

    ranked[cols].to_csv(csv_path, index=False)
    ranked.to_crs("EPSG:4326")[cols + ["geometry"]].to_file(geo_path, driver="GeoJSON")
    prov_path.write_text(json.dumps(provenance, indent=2))
    return [csv_path, geo_path, prov_path]


written = export_results(
    ranked,
    build_provenance(CFG, ENDPOINTS, WEIGHTS, source_note, ranked, fires, RUN_AT),
    RUN_AT,
)
print("wrote:", ", ".join(p.name for p in written) if written else "nothing (no zones)")

## Smoke test

The functions above are pure enough to test without touching the network. This builds
synthetic zones, detections, and perimeters with known answers, then checks that the
joins, the burn overlay, and the scoring behave. Run it after editing anything — it takes
under a second and catches the silent failures (an empty overlay from a CRS mismatch, a
zero-variance component) that otherwise look like a plausible ranking.


In [ ]:
from shapely.geometry import Point, Polygon


def make_fixture_zones(n: int = 4) -> gpd.GeoDataFrame:
    rows = []
    for k in range(n):
        x0 = -122 + k
        rows.append(
            {
                "zone_id": f"TSTZ{k}",
                "zone_name": f"Test zone {k}",
                "state": "CA",
                "zone_type": "fire",
                "n_alerts": 1,
                "events": "Red Flag Warning",
                "severity": "Severe",
                "expires": pd.Timestamp("2030-01-01", tz="UTC"),
                "headline": "test",
                "offices": "NWS Test",
                "hours_remaining": 6.0,
                "geometry": Polygon(
                    [(x0, 36), (x0 + 0.6, 36), (x0 + 0.6, 36.6), (x0, 36.6)]
                ),
            }
        )
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")


def make_fixture_detections(counts=(0, 1, 3, 5)) -> gpd.GeoDataFrame:
    """`counts[k]` detections inside zone k, so the expected join result is known."""
    pts, frp = [], []
    for k, count in enumerate(counts):
        for i in range(count):
            pts.append(Point(-122 + k + 0.1 + 0.05 * i, 36.3))
            frp.append(10.0 * (i + 1))
    return gpd.GeoDataFrame({"frp": frp}, geometry=pts, crs="EPSG:4326")


def make_fixture_perimeters() -> gpd.GeoDataFrame:
    """One fire covering a quarter of zone 3, plus a reburn of the same footprint."""
    box = Polygon([(-119, 36), (-118.7, 36), (-118.7, 36.3), (-119, 36.3)])
    gdf = gpd.GeoDataFrame(
        {
            "event_id": ["A", "B"],
            "incident": ["Alpha", "Bravo"],
            "incid_type": ["Wildfire", "Wildfire"],
            "acres": [1000.0, 2000.0],
            "ig_date": pd.to_datetime(["1995-08-01", "2015-08-01"]),
        },
        geometry=[box, box],
        crs="EPSG:4326",
    )
    gdf["ig_year"] = gdf["ig_date"].dt.year
    return gdf


def run_smoke_test(cfg: Settings, weights: dict) -> None:
    zones, fires_t, mtbs_t = (
        make_fixture_zones(),
        make_fixture_detections(),
        make_fixture_perimeters(),
    )

    # scale01
    assert scale01([0, 0, 0]).tolist() == [0.0, 0.0, 0.0], (
        "constant input must give zeros"
    )
    assert scale01([0, 5, 10]).tolist() == [0.0, 0.5, 1.0]

    # pick_column
    assert pick_column(["Ig_Date", "geometry"], "ig_date") == "Ig_Date"
    assert pick_column(["geometry"], "nope") is None

    # detection join
    z = attach_detection_stats(
        to_equal_area(zones, cfg), to_equal_area(fires_t, cfg), cfg
    )
    assert z["n_detections"].tolist() == [0, 1, 3, 5], z["n_detections"].tolist()
    assert z.loc[z.zone_id == "TSTZ0", "frp_sum"].iloc[0] == 0

    # burn overlay: two overlapping fires, one footprint -> counted twice, measured once
    z = attach_burn_history(z, repair_geometries(to_equal_area(mtbs_t, cfg)), 2026)
    hit = z[z.zone_id == "TSTZ3"].iloc[0]
    assert hit.n_hist_fires == 2, "distinct fires should be counted separately"
    assert 0.2 < hit.burned_frac < 0.3, f"dissolve failed, got {hit.burned_frac:.3f}"
    assert hit.years_since_burn == 11
    assert z[z.zone_id == "TSTZ0"].iloc[0].burned_frac == 0

    # empty inputs must not raise
    assert intersect_zones_with_fires(empty_gdf(["zone_id"]), mtbs_t).empty
    assert summarize_detections(pd.DataFrame(columns=["zone_id", "frp"])).empty
    assert rank_zones(empty_gdf(["zone_id"]), weights).empty
    assert alerts_to_frame([]).empty
    assert zone_urls_by_id(alerts_to_frame([])) == {}

    # ranking: more fire and more history should sort higher
    r = rank_zones(z, weights)
    assert r.iloc[0].zone_id == "TSTZ3", r[["zone_id", "score"]].to_dict("records")
    assert r.iloc[-1].zone_id == "TSTZ0"
    assert abs(r["score"].max() - 100) < 1e-6, "top zone maxes every component"

    print(
        f"smoke test passed — {len(r)} fixture zones scored "
        f"{r['score'].min():.1f}–{r['score'].max():.1f}"
    )


run_smoke_test(CFG, WEIGHTS)

## What this does not tell you

- **A detection is not a fire.** MODIS/VIIRS pick up gas flares, industrial heat, and
  agricultural burning. High-ranking zones need a look at the FIRMS attributes and, where
  available, the AlertCalifornia camera feeds in the catalog.
- **The score is unvalidated.** The weights are a defensible starting point, not a fitted
  model. Nothing here has been checked against subsequent ignitions. Treat the ranking as
  a way to order attention, and change `WEIGHTS` to match what you actually care about.
- **Burn history cuts both ways.** Frequent past fire signals a fire-prone regime; a
  _recent_ burn usually means depleted fuel. That is why `years_since_burn` is reported
  rather than scored.
- **MTBS has a size floor and a reporting lag** — roughly 1,000 acres in the west, 500 in
  the east, and it runs a year or more behind the present. Small and very recent fires are
  simply absent, so `burned_frac` is a floor.
- **Zones are coarse.** A fire weather zone can span a hundred kilometres of very
  different terrain. Zone-level ranking says "look here", not "this ridge".
- **Geography.** `EPSG:5070` is CONUS-appropriate; area figures for Alaska, Hawaii, and
  Puerto Rico will be distorted. The no-auth FIRMS file also covers only CONUS + Hawaii.
- **No exposure layer.** This ranks hazard, not consequence. Two zones with identical
  scores can differ enormously in who lives there.

## Where to take it next, from the same catalog

- **Calibrated hazard** — swap the burn-history heuristic for USFS _Probabilistic Wildfire
  Risk_ burn probability, or _Wildfire Risk to Communities_, both no-auth.
- **Fuel condition** — MODIS or Sentinel-2 NDVI as a live dryness proxy; LANDFIRE seasonal
  fuels for fuel model context. This is the biggest missing driver: the warning tells you
  the weather, not what is there to burn.
- **Exposure and vulnerability** — CarbonPlan building-level risk, CDC/ATSDR SVI, EPA
  EJScreen, and CDC PLACES (asthma/COPD prevalence) turn a hazard rank into an impact
  rank.
- **Downwind effects** — NOAA HMS smoke polygons plus EPA AirNow move the question from
  "where will it burn" to "who is breathing it".
- **Response capacity** — the OSM extract (hydrants, water lines, roads) and DOE EAGLE-I
  outage history speak to whether a place can respond.
- **Ground truth** — WFIGS/NIFC current perimeters to check whether detections correspond
  to a recognised, named incident.

Most of these slot in as one new loader plus one new component in `score_components` —
the `Settings` object and the weights dict are the two places to edit.

## Reproducibility

Every endpoint is resolved from `analysis/fire_datasets.csv` at run time; responses are
cached under `data_cache/` with short TTLs for live layers. Each run writes a provenance
JSON recording endpoints, weights, and counts. Because two of the three layers change by
the hour, **re-running this notebook will not reproduce an earlier result** — the run
timestamp is part of the answer, and the exported CSV/GeoJSON are the artifact of record.
